# Projeto 1 - Sistema Genetico-Neural GA-MLP

Notebook narrativo unico para a segunda avaliacao de Inteligencia Computacional.

Objetivo: documentar e executar o processo completo mantendo os scripts como fonte operacional: EDA, baseline Random Forest/MLP e sistema hibrido Algoritmo Genetico + MLP.

## 1. Contexto da avaliacao

A segunda avaliacao exige um sistema hibrido ou ensemble, combinando tecnicas da ementa. O Projeto 1 usa uma arquitetura Genetico-Neural: o Algoritmo Genetico seleciona features e topologia da MLP; a MLP mede o fitness e depois e treinada com a melhor configuracao encontrada.

Requisitos rastreados em `.specs/features/projeto-1-genetico-neural/`:

- justificar a arquitetura hibrida;
- explicar a comunicacao entre GA e MLP;
- comparar contra baseline simples;
- avaliar com metricas robustas;
- discutir ganho/perda, limitacoes e trabalhos futuros.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

cwd = Path.cwd()
project_candidates = [
    cwd,
    cwd / "projeto_1_genetico_neural",
    cwd.parent,
]
PROJECT_DIR = next(
    candidate for candidate in project_candidates
    if (candidate / "dataset" / "heart_failure_clinical_records_dataset.csv").exists()
)

DATASET_PATH = PROJECT_DIR / "dataset" / "heart_failure_clinical_records_dataset.csv"
REPORTS_DIR = PROJECT_DIR / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_DIR)
print(DATASET_PATH)

## 2. Carregamento e perfil inicial do dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)
display(df.head())
display(df.describe().T)
display(df.isna().sum().rename("missing_values"))
display(df["DEATH_EVENT"].value_counts(normalize=True).rename("target_distribution"))

## 3. EDA

O notebook executa o script `eda_heart_failure.py` para manter a trilha reprodutivel. Em seguida, exibe as figuras geradas.

In [ ]:
subprocess.run([sys.executable, "eda_heart_failure.py"], cwd=PROJECT_DIR, check=True)

In [ ]:
for image_name in ["eda_distribuicoes.png", "eda_correlacao.png"]:
    image_path = PROJECT_DIR / image_name
    if image_path.exists():
        print(image_path)
        display(Image(filename=str(image_path)))

## 4. Baseline: Random Forest e MLP

O baseline compara Random Forest com ajuste de hiperparametros e MLP simples com GridSearchCV. Esse e o ponto de referencia para julgar ganho ou perda do hibrido.

In [ ]:
subprocess.run([sys.executable, "baseline.py"], cwd=PROJECT_DIR, check=True)

Resultados de referencia validados localmente em 2026-07-02:

| Modelo | Accuracy | Precision | Recall | F1 | ROC AUC |
|---|---:|---:|---:|---:|---:|
| Random Forest | 0.8333 | 0.8000 | 0.6316 | 0.7059 | 0.9101 |
| MLP | 0.7167 | 0.5714 | 0.4211 | 0.4848 | 0.7754 |

## 5. Sistema hibrido GA-MLP

O cromossomo codifica duas decisoes:

- bits de selecao de features;
- bits de topologia, convertidos em numero de neuronios ocultos.

A funcao de fitness treina uma MLP e mede F1-Score no conjunto de validacao. O melhor individuo e treinado em treino+validacao e avaliado no teste cego.

In [ ]:
ga_output = TABLES_DIR / "ga_mlp_results.json"
if ga_output.exists():
    print(f"Usando resultado completo existente: {ga_output}")
else:
    ga_output = TABLES_DIR / "ga_mlp_results_notebook_quick.json"
    subprocess.run(
        [
            sys.executable,
            "hybrid_ga_mlp.py",
            "--quick",
            "--seed",
            "42",
            "--output",
            str(ga_output),
        ],
        cwd=PROJECT_DIR,
        check=True,
    )

In [ ]:
with ga_output.open(encoding="utf-8") as fp:
    ga_results = json.load(fp)

comparison_path = TABLES_DIR / "model_comparison.csv"
if comparison_path.exists():
    display(pd.read_csv(comparison_path))

display(pd.DataFrame([ga_results["test_metrics"]]))
display(pd.DataFrame(ga_results["generation_history"]))
display(pd.DataFrame({
    "selected_features": ga_results["selection"]["selected_features"]
}))
ga_results["selection"]

## 6. Leitura critica

A qualidade do Projeto 1 nao depende de o hibrido vencer sempre o baseline. Para a avaliacao, o ponto principal e demonstrar uma arquitetura hibrida coerente, explicar a comunicacao entre os componentes e discutir criticamente o ganho ou perda frente a uma abordagem simples.

Limitacoes esperadas:

- dataset pequeno, com 299 registros;
- risco de overfitting evolutivo no conjunto de validacao;
- custo computacional do fitness, pois cada individuo treina uma MLP;
- necessidade de repetir com multiplas sementes ou K-fold em trabalhos futuros.

## 7. Proximos passos para fechamento

1. Rodar o GA-MLP em modo completo, sem `--quick`.
2. Comparar a tabela final do hibrido com RF e MLP.
3. Atualizar o relatorio tecnico do Projeto 1.
4. Preparar a apresentacao de 15 minutos com arquitetura, resultados e limitacoes.